# E-commerce Conversion Analytics and Purchase Propensity ML

**A self-contained, reproducible portfolio notebook**

This notebook embeds the complete deployment-safe session-level modeling dataset, reconstructs the business analysis, trains two leakage-safe machine-learning models, and generates every chart in one run. No external CSV, JSON, image, SQL, or Python file is required.

## Executive summary

- 310,014 reconstructed sessions produced 4,247 purchases and a 1.37% conversion rate.
- The largest funnel loss occurs before product discovery.
- A chronological holdout evaluates ranking performance on future sessions.
- Hashed logistic regression reaches approximately 2.0x lift in the highest-scored 10% of sessions.
- The score is suitable for prioritizing experiments, not for making causal claims or denying service.

Use **Run All** from a fresh kernel. The first cell installs a missing package only when necessary.

## 1. Environment setup

The notebook uses Python, NumPy, pandas, and Matplotlib. The bootstrap is idempotent: it does nothing when the libraries are already installed.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import base64
import gzip
import hashlib
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#0F766E", "#D97706", "#7C3AED", "#DC2626", "#64748B"]
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | NumPy {np.__version__}")

## 2. Embedded data asset

The text below is a gzip-compressed, Base64-encoded CSV containing all 310,014 session rows and only the fields permitted at scoring time plus the outcome. Embedding the reviewed dataset makes the notebook portable while preserving a strict leakage boundary.

In [ ]:
EMBEDDED_SESSION_DATA_GZIP_BASE64 = """..."""

compressed = base64.b64decode(EMBEDDED_SESSION_DATA_GZIP_BASE64)
session_data = gzip.decompress(compressed)
df = pd.read_csv(io.BytesIO(session_data), parse_dates=["session_date"])

EXPECTED_ROWS = 310_014
EXPECTED_COLUMNS = 9
assert df.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS)
assert df["converted"].isin([0, 1]).all()
print(f"Loaded {len(df):,} sessions from the embedded asset ({len(compressed) / 1_000_000:.2f} MB compressed).")
display(df.head())

## 3. Data quality and feature contract

The model is scored at session start. Therefore, the feature set deliberately excludes page views, product interactions, cart and checkout events, purchases, transactions, revenue, engagement, event counts, and session duration.

In [ ]:
LABEL = "converted"
DATE = "session_date"
TEST_START = pd.Timestamp("2021-01-11")
SAFE_FEATURES = [
    "day_of_week",
    "is_weekend",
    "traffic_source",
    "traffic_medium",
    "device_category",
    "operating_system",
    "country",
]
PROHIBITED = {
    "total_events", "page_views", "product_views", "add_to_cart_events",
    "checkout_events", "payment_events", "purchase_events", "revenue",
    "transaction_count", "session_duration_seconds", "engagement_time_seconds",
}

assert set(SAFE_FEATURES).isdisjoint(PROHIBITED)
assert not df[SAFE_FEATURES + [DATE, LABEL]].isna().any().any()
quality = pd.DataFrame({
    "check": ["Rows", "Columns", "Missing values", "Complete rows", "Start date", "End date"],
    "result": [
        f"{len(df):,}", len(df.columns), int(df.isna().sum().sum()),
        int(df.notna().all(axis=1).sum()), df[DATE].min().date(), df[DATE].max().date(),
    ],
})
display(quality)
print("Leakage audit: PASSED")

## 4. Business performance and conversion funnel

In [ ]:
sessions = len(df)
purchases = int(df[LABEL].sum())
conversion_rate = float(df[LABEL].mean())
revenue = 315_948.00

kpis = pd.DataFrame({
    "KPI": ["Sessions", "Purchases", "Conversion rate", "Recorded revenue", "Revenue per session"],
    "Value": [f"{sessions:,}", f"{purchases:,}", f"{conversion_rate:.2%}", f"${revenue:,.0f}", f"${revenue/sessions:.2f}"],
})
display(kpis)

funnel = pd.DataFrame({
    "stage": ["Visit", "Product View", "Add to Cart", "Checkout", "Payment", "Purchase"],
    "sessions": [310_014, 65_393, 15_187, 9_086, 5_875, 4_247],
})
funnel["stage_conversion"] = funnel["sessions"] / funnel["sessions"].shift(1)
funnel.loc[0, "stage_conversion"] = 1.0
funnel["drop_off"] = funnel["sessions"].shift(1) - funnel["sessions"]
funnel_display = funnel.copy()
funnel_display["sessions"] = funnel_display["sessions"].map("{:,}".format)
funnel_display["stage_conversion"] = funnel_display["stage_conversion"].map("{:.2%}".format)
funnel_display["drop_off"] = funnel_display["drop_off"].fillna(0).map("{:,.0f}".format)
display(funnel_display)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(funnel["stage"], funnel["sessions"], color=COLORS)
ax.invert_yaxis()
ax.set_title("Conversion Funnel by Session")
ax.set_xlabel("Sessions")
for bar, value in zip(bars, funnel["sessions"]):
    ax.text(value + 4000, bar.get_y() + bar.get_height()/2, f"{value:,}", va="center")
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

### Funnel interpretation

The Visit-to-Product-View transition loses 244,621 sessions, the largest absolute leak. Product View-to-Add-to-Cart loses another 50,206 sessions. Payment-to-Purchase retains 72.29%, so discovery and product-detail experiments should be prioritized before a checkout-first redesign.

In [ ]:
daily = df.groupby(DATE).agg(sessions=(LABEL, "size"), purchases=(LABEL, "sum")).reset_index()
daily["conversion_rate"] = daily["purchases"] / daily["sessions"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(daily[DATE], daily["sessions"], color=COLORS[0], linewidth=1.5)
axes[0].set_title("Daily Session Volume")
axes[0].set_ylabel("Sessions")
axes[1].plot(daily[DATE], daily["conversion_rate"] * 100, color=COLORS[1], linewidth=1.5)
axes[1].set_title("Daily Conversion Rate")
axes[1].set_ylabel("Conversion rate (%)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Machine-learning methodology

The test period follows the training period, approximating a real deployment. The primary metric is PR-AUC because purchases are rare. ROC-AUC, log loss, Brier score, top-decile recall, and lift provide complementary views of discrimination, calibration, and operational usefulness.

In [ ]:
train = df[df[DATE] < TEST_START].copy()
test = df[df[DATE] >= TEST_START].copy()
y_train = train[LABEL].astype(int).to_numpy()
y_test = test[LABEL].astype(int).to_numpy()

split_summary = pd.DataFrame({
    "sample": ["Train", "Test"],
    "start": [train[DATE].min().date(), test[DATE].min().date()],
    "end": [train[DATE].max().date(), test[DATE].max().date()],
    "sessions": [len(train), len(test)],
    "purchases": [int(y_train.sum()), int(y_test.sum())],
    "conversion_rate": [y_train.mean(), y_test.mean()],
})
assert len(train) == 228_487 and len(test) == 81_527
split_display = split_summary.copy()
split_display["sessions"] = split_display["sessions"].map("{:,}".format)
split_display["purchases"] = split_display["purchases"].map("{:,}".format)
split_display["conversion_rate"] = split_display["conversion_rate"].map("{:.2%}".format)
display(split_display)

In [ ]:
def stable_bucket(value, column, buckets):
    raw = f"{column}={value}".encode("utf-8", errors="replace")
    return int.from_bytes(hashlib.blake2b(raw, digest_size=8).digest(), "little") % buckets

def hash_matrix(frame, buckets):
    matrices = []
    for column in SAFE_FEATURES:
        values = frame[column].fillna("<MISSING>").astype(str)
        mapping = {value: stable_bucket(value, column, buckets) for value in values.unique()}
        matrices.append(values.map(mapping).to_numpy(dtype=np.int32))
    return np.column_stack(matrices)

def sigmoid(z):
    z = np.clip(z, -35.0, 35.0)
    return 1.0 / (1.0 + np.exp(-z))

def fit_hashed_logistic(x_idx, y, buckets=2048, epochs=30, batch_size=8192, learning_rate=0.08, l2=1e-5):
    rng = np.random.default_rng(20260729)
    w = np.zeros(buckets, dtype=np.float64)
    b = float(np.log(y.mean() / (1.0 - y.mean())))
    mw, vw = np.zeros_like(w), np.zeros_like(w)
    mb = vb = 0.0
    beta1, beta2, eps, step = 0.9, 0.999, 1e-8, 0
    history = []
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for start in range(0, len(y), batch_size):
            ids = order[start:start + batch_size]
            xb, yb = x_idx[ids], y[ids]
            p = sigmoid(b + w[xb].sum(axis=1))
            err = p - yb
            grad = np.zeros_like(w)
            for j in range(xb.shape[1]):
                np.add.at(grad, xb[:, j], err)
            grad = grad / len(ids) + l2 * w
            grad_b = float(err.mean())
            step += 1
            mw, vw = beta1 * mw + (1-beta1) * grad, beta2 * vw + (1-beta2) * grad * grad
            mb, vb = beta1 * mb + (1-beta1) * grad_b, beta2 * vb + (1-beta2) * grad_b * grad_b
            w -= learning_rate * (mw / (1-beta1**step)) / (np.sqrt(vw / (1-beta2**step)) + eps)
            b -= learning_rate * (mb / (1-beta1**step)) / (np.sqrt(vb / (1-beta2**step)) + eps)
        p_all = sigmoid(b + w[x_idx].sum(axis=1))
        history.append(float(-np.mean(y*np.log(np.clip(p_all, 1e-12, 1)) + (1-y)*np.log(np.clip(1-p_all, 1e-12, 1)))))
    return w, b, history

In [ ]:
def fit_categorical_naive_bayes(frame, y, alpha=10.0):
    model = {"prior": float((y.sum()+alpha)/(len(y)+2*alpha)), "columns": {}}
    for column in SAFE_FEATURES:
        values = frame[column].fillna("<MISSING>").astype(str)
        table = pd.crosstab(values, y)
        for cls in [0, 1]:
            if cls not in table.columns:
                table[cls] = 0
        n_values, probs, defaults = len(table)+1, {}, {}
        for cls in [0, 1]:
            denom = float((y == cls).sum() + alpha*n_values)
            probs[str(cls)] = ((table[cls]+alpha)/denom).to_dict()
            defaults[str(cls)] = alpha/denom
        model["columns"][column] = {"probs": probs, "defaults": defaults}
    return model

def predict_categorical_naive_bayes(model, frame):
    prior = model["prior"]
    log0, log1 = np.full(len(frame), np.log(1-prior)), np.full(len(frame), np.log(prior))
    for column in SAFE_FEATURES:
        values, spec = frame[column].fillna("<MISSING>").astype(str), model["columns"][column]
        log0 += np.log(values.map(spec["probs"]["0"]).fillna(spec["defaults"]["0"]).to_numpy())
        log1 += np.log(values.map(spec["probs"]["1"]).fillna(spec["defaults"]["1"]).to_numpy())
    return sigmoid(log1-log0)

def average_precision(y, p):
    ys = y[np.argsort(-p, kind="mergesort")]
    return float(((np.cumsum(ys)/np.arange(1, len(ys)+1))*ys).sum()/ys.sum())

def roc_auc(y, p):
    order, ranks, sorted_p = np.argsort(p, kind="mergesort"), np.empty(len(p)), np.sort(p, kind="mergesort")
    i = 0
    while i < len(p):
        j = i + 1
        while j < len(p) and sorted_p[j] == sorted_p[i]:
            j += 1
        ranks[order[i:j]] = (i + 1 + j) / 2
        i = j
    n1, n0 = y.sum(), len(y)-y.sum()
    return float((ranks[y == 1].sum()-n1*(n1+1)/2)/(n1*n0))

def evaluate(y, p, capacity=0.10):
    clipped = np.clip(p, 1e-12, 1-1e-12)
    n_top, base = int(np.ceil(len(y)*capacity)), float(y.mean())
    top = np.argsort(-p)[:n_top]
    return {
        "PR-AUC": average_precision(y, p),
        "ROC-AUC": roc_auc(y, p),
        "Log loss": float(-np.mean(y*np.log(clipped)+(1-y)*np.log(1-clipped))),
        "Brier score": float(np.mean((p-y)**2)),
        "Top-10% recall": float(y[top].sum()/y.sum()),
        "Top-10% lift": float(y[top].mean()/base),
    }

## 6. Train and compare models

In [ ]:
HASH_BUCKETS = 2048
x_train, x_test = hash_matrix(train, HASH_BUCKETS), hash_matrix(test, HASH_BUCKETS)
weights, intercept, training_loss = fit_hashed_logistic(x_train, y_train, HASH_BUCKETS)
p_logistic = sigmoid(intercept + weights[x_test].sum(axis=1))

nb_model = fit_categorical_naive_bayes(train, y_train)
p_nb = predict_categorical_naive_bayes(nb_model, test)

probabilities = {
    "Hashed logistic regression": p_logistic,
    "Categorical Naive Bayes": p_nb,
}
results = pd.DataFrame({name: evaluate(y_test, p) for name, p in probabilities.items()}).T
selected_model = results["PR-AUC"].idxmax()
display(results.round(4))
print(f"Selected by PR-AUC: {selected_model}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results[["PR-AUC", "ROC-AUC"]].plot(kind="bar", ax=axes[0], color=COLORS[:2])
axes[0].set_title("Model Ranking Metrics")
axes[0].set_ylim(0, 0.7)
results[["Top-10% recall", "Top-10% lift"]].plot(kind="bar", ax=axes[1], color=COLORS[2:4])
axes[1].set_title("Operational Targeting Metrics")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def decile_table(y, p):
    order = np.argsort(-p, kind="mergesort")
    decile = np.empty(len(y), dtype=int)
    decile[order] = np.minimum(10, np.floor(np.arange(len(y))*10/len(y)).astype(int)+1)
    table = pd.DataFrame({"converted": y, "score": p, "score_decile": decile})
    base = table["converted"].mean()
    out = table.groupby("score_decile").agg(
        sessions=("converted", "size"), purchases=("converted", "sum"),
        avg_score=("score", "mean"), observed_rate=("converted", "mean"),
    ).reset_index()
    out["lift_vs_average"] = out["observed_rate"]/base
    return out

lift = decile_table(y_test, p_logistic)
lift_display = lift.copy()
lift_display["sessions"] = lift_display["sessions"].map("{:,}".format)
lift_display["purchases"] = lift_display["purchases"].map("{:,}".format)
lift_display["avg_score"] = lift_display["avg_score"].map("{:.2%}".format)
lift_display["observed_rate"] = lift_display["observed_rate"].map("{:.2%}".format)
lift_display["lift_vs_average"] = lift_display["lift_vs_average"].map("{:.2f}x".format)
display(lift_display)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(lift["score_decile"].astype(str), lift["lift_vs_average"], color=COLORS[0])
ax.axhline(1, color="#334155", linestyle="--", linewidth=1)
ax.set(title="Hashed Logistic Regression Lift by Score Decile", xlabel="Score decile (1 = highest risk)", ylabel="Lift vs. average")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def calibration_table(y, p):
    order = np.argsort(p, kind="mergesort")
    band = np.empty(len(y), dtype=int)
    band[order] = np.minimum(10, np.floor(np.arange(len(y))*10/len(y)).astype(int)+1)
    table = pd.DataFrame({"converted": y, "score": p, "risk_band": band})
    return table.groupby("risk_band").agg(
        sessions=("converted", "size"),
        predicted_rate=("score", "mean"),
        observed_rate=("converted", "mean"),
    ).reset_index()

calibration = calibration_table(y_test, p_logistic)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(calibration["predicted_rate"]*100, calibration["observed_rate"]*100, marker="o", color=COLORS[1], label="Model")
limit = max(calibration["predicted_rate"].max(), calibration["observed_rate"].max())*100*1.1
ax.plot([0, limit], [0, limit], linestyle="--", color="#64748B", label="Perfect calibration")
ax.set(title="Calibration by Equal-Frequency Risk Band", xlabel="Predicted conversion rate (%)", ylabel="Observed conversion rate (%)")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Interpretation and recommended action

The model provides real but modest ranking signal. The highest-scored 10% captures about 20% of future purchases, roughly doubling purchase density relative to the holdout average. Because session-start variables cannot represent product intent or in-session behavior, this ceiling is expected.

### Recommended experiment

Run a randomized controlled trial among eligible sessions:

- **Population:** new sessions eligible for a product-discovery intervention.
- **Treatment:** improved navigation, product recommendations, or a landing-page variant.
- **Control:** current experience.
- **Primary metric:** purchase conversion rate.
- **Guardrails:** page latency, bounce rate, refund rate, and revenue per visitor.
- **Segmentation:** report treatment effects by model score band, but preserve random assignment within each band.

The model should prioritize measurement opportunities. It should not be interpreted as causal, used to exclude customers, or deployed without drift and calibration monitoring.

## 8. Reproducibility checks

These assertions fail loudly if the embedded data, chronological split, class counts, or expected model behavior changes.

In [ ]:
assert len(df) == 310_014
assert int(df[LABEL].sum()) == 4_247
assert len(train) == 228_487 and int(y_train.sum()) == 3_335
assert len(test) == 81_527 and int(y_test.sum()) == 912
assert 0.015 < results.loc["Hashed logistic regression", "PR-AUC"] < 0.020
assert 1.9 < results.loc["Hashed logistic regression", "Top-10% lift"] < 2.1
assert selected_model == "Hashed logistic regression"
print("All reproducibility checks passed. The notebook is ready for portfolio use.")

## 9. Limitations

- The raw GA4 event export is not embedded; the compact session-level analytical asset is embedded instead.
- Funnel stage totals are reviewed aggregates from event reconstruction, while model rows contain only score-time-safe fields.
- The observation window is short and includes seasonal behavior.
- Performance should be revalidated on later data before production use.
- Predictive lift does not establish that an intervention will create incremental purchases.

## Conclusion

This project combines funnel analytics, a formal leakage contract, chronological validation, rare-event model evaluation, calibration analysis, and experiment design in one portable notebook. It demonstrates an end-to-end analytics workflow while keeping claims proportional to the evidence.